# Bombcell workshop – Open Lab Online, 20 October 2026

**Goal:** go from a Kilosort output folder to a curated set of units, and understand *why* each unit got its label.

1. Run Bombcell
2. What the output looks like
3. Why curate? How unit selection changes your results
4. It depends on your question: single neurons vs. populations
5. Look at units in the GUI
6. Combining with Phy
7. Setting thresholds

Further reading: other metrics and options (sliding refractory period, distance metrics and drift, time chunks)

**Data:** one session from the [chronic Neuropixels 2.0 dataset](https://doi.org/10.5522/04/24411841.v2)
(Lebedeva, Okun, Krumin, Carandini; Steinmetz et al. *Science* 2021; CC BY-NC-SA 4.0): mouse visual cortex,
spike-sorted with pyKilosort, while the mouse viewed 112 natural images, 5 repeats each.

## 0. Setup

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # locally, uv has already installed everything
    !pip install -q bombcell

In [2]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bombcell as bc

URL = "https://ndownloader.figshare.com/files/61060780"   # AL031_2019-12-13_noRawData.tar (~1.7 GB)
DATA_DIR = Path("/content/data") if IN_COLAB else Path("../data").resolve()  # where the dataset goes
FS = 30000                                                # sampling rate (Hz)

✅ ipywidgets available - interactive GUI ready


In [3]:
# Download and extract (~1.7 GB, a few minutes). Skipped if already there.
DATA_DIR.mkdir(parents=True, exist_ok=True)
if not any(DATA_DIR.rglob("spike_times.npy")):
    !curl -sL {URL} | tar -x -C {DATA_DIR}

ks_dir = next(DATA_DIR.rglob("spike_times.npy")).parent
session_dir = ks_dir.parent
save_path = session_dir / "bombcell"
print("kilosort folder:", ks_dir)

kilosort folder: /Users/jf5479/Dropbox/Python/Bombcell_OpenLabOnline_course_2026/data/AL031_2019-12-13/ks


## 1. Run Bombcell

Bombcell needs only the Kilosort folder. The raw data (`.ap.bin`) is optional: it improves the
amplitude and SNR metrics, but here we run on templates only.

All settings live in one dictionary, `param`.

*Using SpikeInterface?* Bombcell also runs directly on a SpikeInterface `SortingAnalyzer`: see this
[demo notebook](https://github.com/Julie-Fabre/bombcell/blob/main/py_bombcell/demos/BC_demo_spikeinterface.ipynb).

In [4]:
param = bc.get_default_parameters(str(ks_dir), kilosort_version=2)   # pyKilosort ~ Kilosort 2.5
param["computeDrift"] = 1                # estimate per-unit drift
param["computeDistanceMetrics"] = 1      # isolation distance / L-ratio: slower (~1 min here), needed for part 7
param["splitGoodAndMua_NonSomatic"] = 0  # 1 = split non-somatic units into good / MUA

# the classification thresholds
bc.print_qm_thresholds(param) if hasattr(bc, "print_qm_thresholds") else \
    print({k: v for k, v in param.items() if k.startswith(("min", "max"))})

Current threshold params:
maxNPeaks = 2, maxNTroughs = 1
minWvDuration = 100, maxWvDuration = 1150
maxWvBaselineFraction = 0.3, minSpatialDecaySlope = -0.008
maxPercSpikesMissing = 20, minNumSpikes = 300
maxRPVviolations = 0.1, minPresenceRatio = 0.7
minAmplitude = 40, minSNR = 5
maxDrift = 100


In [ ]:
quality_metrics, param, unit_type, unit_type_string = bc.run_bombcell(str(ks_dir), str(save_path), param)

🚀 Starting BombCell quality metrics pipeline...
📁 Processing data from: /Users/jf5479/Dropbox/Python/Bombcell_OpenLabOnline_course_2026/data/AL031_2019-12-13/ks
Results will be saved to: /Users/jf5479/Dropbox/Python/Bombcell_OpenLabOnline_course_2026/data/AL031_2019-12-13/bombcell

Loading ephys data...
Loaded ephys data: 206 units, 1,933,635 spikes

⚙️ Computing quality metrics for 206 units...
   (Progress bar will appear below)


Computing bombcell quality metrics:   0%|          | 0/206 units

**Reading the summary plots** (more in the [wiki](https://github.com/Julie-Fabre/bombcell/wiki/Summary-output-plots)):
- **Waveform overlay:** templates of each class. Do the "noise" ones look like noise?
- **Upset plots:** which metrics exclude which units, and how they overlap.
- **Histograms:** distribution of each metric with its threshold. Is a threshold cutting through the middle of a distribution, or removing almost nothing?

## 2. What the output looks like

In [ ]:
# Bombcell's own output folder
for f in sorted(save_path.rglob("*")):
    if f.is_file():
        print(f"{f.stat().st_size / 1e3:8.0f} kB  {f.relative_to(save_path)}")

In [ ]:
# ...and one .tsv per metric in the Kilosort folder, so Phy shows them as columns.
# cluster_bc_unitType.tsv holds the labels.
print(sorted(f.name for f in ks_dir.glob("cluster_*.tsv")))
pd.read_csv(ks_dir / "cluster_bc_unitType.tsv", sep="\t").head()

In [ ]:
# One row per unit: label + every metric
qm_table = pd.DataFrame(quality_metrics)
qm_table.insert(0, "bc_label", unit_type_string)
print(qm_table["bc_label"].value_counts())
qm_table.head()

In [ ]:
# Same table, but True/False: does each unit pass each threshold?
bc.make_qm_table(quality_metrics, param, unit_type_string).head()

In [ ]:
# Reload everything later without recomputing
param, quality_metrics, _ = bc.load_bc_results(str(save_path))

In [ ]:
# A methods paragraph for your paper, with the exact parameters you used
bc.print_methods_text(param, quality_metrics)

## 3. Why curate? How unit selection changes your results

Quality metrics only matter if they change what you conclude. Here we compute a few numbers a paper might report
about these neurons and see how they depend on which units you keep.

The mouse saw 112 natural images, 5 times each. For each unit we compute its **stimulus information** (bits/s):
how much its firing rate tells you about which image is on the screen. Units with few spikes or random firing
get a positive value by chance, so we subtract the value obtained after shuffling the image labels. Values near zero mean "no information".

In [ ]:
from scipy.io import loadmat

def load_mat(path):  # MATLAB file with a single variable
    m = loadmat(path)
    return next(v for k, v in m.items() if not k.startswith("__")).squeeze()

stim_times = load_mat(session_dir / "stimTimes.mat")         # image onsets (s)
stim_ids = load_mat(session_dir / "stimIDs.mat").astype(int)  # which image (1-112)
print(f"{len(stim_times)} presentations of {len(np.unique(stim_ids))} images")

In [ ]:
WINDOW = (0.05, 1.0)  # response window after image onset (s); images are shown for 1 s

spike_times = np.load(ks_dir / "spike_times.npy").squeeze() / FS
spike_units = np.load(ks_dir / "spike_templates.npy").squeeze()
unit_ids = qm_table["phy_clusterID"].to_numpy().astype(int)

# firing rate of every unit on every presentation: units x presentations
rates = np.zeros((len(unit_ids), len(stim_times)))
for k, u in enumerate(unit_ids):
    st = spike_times[spike_units == u]
    rates[k] = np.searchsorted(st, stim_times + WINDOW[1]) - np.searchsorted(st, stim_times + WINDOW[0])
rates /= WINDOW[1] - WINDOW[0]

def stim_info(rates, ids):
    # Skaggs information about image identity (bits/s): mean over images of r_i * log2(r_i / r)
    _, idx = np.unique(ids, return_inverse=True)
    r_img = np.stack([np.bincount(idx, weights=r) for r in rates]) / np.bincount(idx)
    r_mean = r_img.mean(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = np.where(r_img > 0, r_img * np.log2(r_img / r_mean), 0)
    return terms.mean(axis=1)

rng = np.random.default_rng(0)
shuffled = np.mean([stim_info(rates, rng.permutation(stim_ids)) for _ in range(100)], axis=0)
qm_table["stim_info"] = stim_info(rates, stim_ids) - shuffled
qm_table.groupby("bc_label")["stim_info"].describe()

Now the same summary statistics under different unit selections.
"Responsive" = more stimulus information than in all 100 label shuffles.

In [ ]:
shuffle_infos = np.stack([stim_info(rates, rng.permutation(stim_ids)) for _ in range(100)])
qm_table["responsive"] = (stim_info(rates, stim_ids) > shuffle_infos).all(axis=0)
duration = spike_times.max()
qm_table["firing_rate"] = qm_table["nSpikes"] / duration

selections = {
    "all units": np.ones(len(qm_table), bool),
    "Bombcell GOOD": (qm_table["bc_label"] == "GOOD").to_numpy(),
    "Bombcell GOOD + MUA": qm_table["bc_label"].isin(["GOOD", "MUA"]).to_numpy(),
}
summary = pd.DataFrame({
    name: {
        "n units": sel.sum(),
        "median firing rate (Hz)": qm_table.loc[sel, "firing_rate"].median(),
        "% responsive": 100 * qm_table.loc[sel, "responsive"].mean(),
    } for name, sel in selections.items()
}).T
summary.round(2)

In [ ]:
# The same table as bar charts: one panel per number you might report
fig, axs = plt.subplots(1, len(summary.columns), figsize=(4 * len(summary.columns), 3.5), sharey=True)
names = summary.index
colors = ["gray", "tab:green", "tab:orange", "tab:blue"]
for ax, col in zip(axs, summary.columns):
    ax.barh(names, summary[col], color=colors[: len(names)])
    ax.set_title(col)
axs[0].invert_yaxis()
plt.tight_layout(); plt.show()

**Discussion:** which numbers change most with curation? Would a reviewer get a different picture of these neurons from each row?

## 4. It depends on your question: single neurons vs. populations

A multi-unit cluster contains spikes from several neurons. Is that a problem? It depends on what you want to say.

**Single-neuron claims** (spike timing, bursting, tuning of *one* cell): a single neuron cannot fire twice within ~2 ms,
so its autocorrelogram has a gap around zero. Below, the most informative GOOD unit next to the stimulus-responsive MUA unit
whose gap is most filled in. That MUA unit responds to the images, but whatever you measure about "its" spike timing or tuning
describes a mixture of neurons.

In [ ]:
def autocorrelogram(st, window=0.05, bin_size=0.001):
    # histogram of time differences between every spike and the spikes after it (mirrored), in Hz
    diffs = []
    for k in range(1, len(st)):
        d = st[k:] - st[:-k]
        d = d[d < window]
        if len(d) == 0:
            break
        diffs.append(d)
    diffs = np.concatenate(diffs) if diffs else np.array([])
    bins = np.arange(-window, window + bin_size, bin_size)
    counts = np.histogram(np.concatenate([diffs, -diffs]), bins)[0]
    return bins[:-1] + bin_size / 2, counts / len(st) / bin_size

def image_responses(k):
    # mean firing rate to each image, sorted, with the standard error over repeats
    _, idx = np.unique(stim_ids, return_inverse=True)
    resp = np.stack([rates[k, idx == i] for i in range(idx.max() + 1)])
    order = np.argsort(resp.mean(1))[::-1]
    return resp.mean(1)[order], resp.std(1)[order] / np.sqrt(resp.shape[1])

def gap_filled(k):
    # autocorrelogram within 1.5 ms of zero, relative to its level at 10-50 ms (0 = clean gap, 1 = no gap)
    lags, acg = autocorrelogram(spike_times[spike_units == unit_ids[k]])
    return acg[np.abs(lags) < 0.0015].mean() / acg[np.abs(lags) > 0.01].mean()

good = qm_table.index[qm_table["bc_label"] == "GOOD"]
responsive_mua = qm_table.index[(qm_table["bc_label"] == "MUA") & qm_table["responsive"]]
examples = {"GOOD": qm_table.loc[good, "stim_info"].idxmax(),
            "MUA": max(responsive_mua, key=gap_filled)}

fig, axs = plt.subplots(2, 2, figsize=(11, 6))
for row, (label, k) in enumerate(examples.items()):
    u = qm_table.loc[k]
    lags, acg = autocorrelogram(spike_times[spike_units == unit_ids[k]])
    axs[row, 0].bar(lags * 1000, acg, width=1, color="tab:green" if label == "GOOD" else "tab:orange")
    axs[row, 0].axvspan(-2, 2, color="gray", alpha=0.2)
    axs[row, 0].set_title(f"{label} unit {int(u.phy_clusterID)}: refractory gap {gap_filled(k):.0%} filled")
    axs[row, 0].set_xlabel("lag (ms)"); axs[row, 0].set_ylabel("rate (Hz)")
    m, se = image_responses(k)
    axs[row, 1].errorbar(np.arange(len(m)), m, se, fmt=".", ms=3, color="k", elinewidth=0.5)
    axs[row, 1].set_title(f"responses to the 112 images (sorted), {u.stim_info:.2f} bits/s")
    axs[row, 1].set_xlabel("image rank"); axs[row, 1].set_ylabel("firing rate (Hz)")
plt.tight_layout(); plt.show()

**Population claims** (what does the population encode?): spikes from several neurons can still carry information.
Can we tell which image is on the screen from the population response? We train on 4 repeats and test on the 5th
(nearest-centroid decoder; chance = 1/112 ≈ 0.9 %). To compare *per unit*, we draw the same number of units from each class.

In [ ]:
def decode(rates, ids):
    # leave-one-repeat-out nearest-centroid decoder; returns fraction of correct image guesses
    z = (rates - rates.mean(1, keepdims=True)) / (rates.std(1, keepdims=True) + 1e-9)
    images, idx = np.unique(ids, return_inverse=True)
    rep = np.zeros(len(ids), int)
    for i in range(len(images)):
        rep[idx == i] = np.arange((idx == i).sum())
    correct = []
    for r in np.unique(rep):
        train, test = rep != r, rep == r
        centroids = np.stack([z[:, train & (idx == i)].mean(1) for i in range(len(images))], axis=1)
        guess = np.argmax(np.corrcoef(z[:, test].T, centroids.T)[: test.sum(), test.sum():], axis=1)
        correct.append(guess == idx[test])
    return np.concatenate(correct).mean()

n = 10  # units drawn per class
draws = {}
for label in ["GOOD", "MUA", "NOISE"]:
    pool = np.flatnonzero(qm_table["bc_label"] == label)
    draws[label] = [decode(rates[rng.choice(pool, n, replace=False)], stim_ids) for _ in range(50)]
whole = {name: decode(rates[sel], stim_ids) for name, sel in selections.items()}

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].boxplot(list(draws.values()))
axs[0].set_xticks([1, 2, 3], list(draws.keys()))
axs[0].axhline(1 / 112, c="gray", ls=":"); axs[0].set_ylabel("decoding accuracy")
axs[0].set_title(f"{n} random units from each class")
axs[1].barh(list(whole.keys()), list(whole.values()), color="tab:blue")
axs[1].axvline(1 / 112, c="gray", ls=":"); axs[1].set_xlabel("decoding accuracy")
axs[1].set_title("all units in each selection"); axs[1].invert_yaxis()
plt.tight_layout(); plt.show()

**Discussion**
- Per unit, GOOD units carry far more information than MUA. Yet whole selections don't simply get better as you add units:
  here, adding MUA to the GOOD units *improves* decoding, while adding noise units on top *hurts* it.
- ✏️ You want to report single-neuron spike timing (e.g. cross-correlograms between pairs). Which thresholds would you tighten?
  Now you want to decode the image from the population: which would you relax?
- There is no single "right" set of units: choose the selection that fits your question, and report it.

## 5. Look at units in the GUI

Browse units by class with the buttons. For each unit, ask: does the waveform look neuronal?
Is there a refractory period in the autocorrelogram? Is the amplitude stable over time, or cut off?

You can also label units yourself; Bombcell saves your labels so you can compare with its own.

In [ ]:
try:  # needed for widgets in Colab
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

gui = bc.unit_quality_gui(
    ks_dir=str(ks_dir),
    quality_metrics=quality_metrics,
    unit_types=unit_type,
    param=param,
    save_path=str(save_path),
)

✏️ **Exercise:** find one unit you disagree with Bombcell about. Which metric drove its label?
Use `qm_table.loc[qm_table.phy_clusterID == UNIT_ID]` to look at its numbers.

In [ ]:
UNIT_ID = 0  # change me
qm_table.loc[qm_table.phy_clusterID == UNIT_ID].T

✏️ **Exercise:** many units here are labelled noise *only* because their spatial decay slope is below `minSpatialDecaySlopeExp`.
This recording uses a Neuropixels 2.0 probe (15 µm between rows). Look at a few of these units in the GUI: are they noise?

## 6. Combining with Phy

Bombcell writes its labels and every metric as `cluster_*.tsv` files into the Kilosort folder (see part 2).
[Phy](https://github.com/cortex-lab/phy) reads these automatically, so when you open the folder in Phy,
`bc_unitType` and each metric appear as columns in the cluster view. You can sort and filter by them,
and use Bombcell's labels as a starting point for manual curation.

![Phy's cluster view with Bombcell's quality metrics as columns](https://github.com/Julie-Fabre/bombcell/assets/29582008/42dd14ce-1d07-4743-8a34-d7604edc5c99)

*Phy's cluster view with Bombcell metrics as columns (from the [Bombcell wiki](https://github.com/Julie-Fabre/bombcell/wiki/Compatibility-with-phy)).*

Phy needs a desktop, so run this on your own computer, not in Colab:

```shell
cd path/to/kilosort_folder
phy template-gui params.py
```

For this dataset, first set `dat_path` in `params.py` to the raw data file (see the dataset's readme).
Without raw data, Phy still opens, but shows no raw traces.

## 7. Setting thresholds

Thresholds are a choice, not a truth. Relabelling is instant: no need to recompute the metrics.

In [ ]:
def relabel(**new_thresholds):
    p = dict(param, **new_thresholds)
    _, labels = bc.get_quality_unit_type(p, quality_metrics)
    return pd.Series(labels).value_counts()

pd.DataFrame({
    "default": relabel(),
    "strict RPV (5%)": relabel(maxRPVviolations=0.05),
    "lenient RPV (20%)": relabel(maxRPVviolations=0.2),
    "presence > 0.9": relabel(minPresenceRatio=0.9),
    "spatial decay > 0.005": relabel(minSpatialDecaySlopeExp=0.005),
}).fillna(0).astype(int)

In [ ]:
# try your own
relabel(maxRPVviolations=0.1, maxPercSpikesMissing=20, minPresenceRatio=0.7, minNumSpikes=300)

**Choosing thresholds from the data.** How do you pick a threshold that isn't arbitrary? One approach (Harris et al., *Nat. Neurosci.* 2016):
plot something you care about, here each unit's stimulus information from part 3, against a quality metric.
If the curve rises and then **plateaus**, units past the plateau are as informative as the best-isolated ones:
a sensible threshold sits where the curve stops rising.

In [ ]:
def plot_vs_info(metric, threshold=None, ax=None, log=False, window=21):
    """Each unit's stimulus information against a quality metric, with a running mean (red)."""
    ax = ax or plt.subplots(figsize=(4, 4))[1]
    d = qm_table[[metric, "stim_info"]].replace([np.inf, -np.inf], np.nan).dropna().sort_values(metric)
    ax.scatter(d[metric], d["stim_info"], s=6, c="tab:blue")
    running = d["stim_info"].rolling(window, center=True, min_periods=window // 2).mean()
    ax.plot(d[metric], running, c="red", lw=3)
    if threshold is not None:
        ax.axvline(threshold, c="k", ls="--", lw=1)
    if log:
        ax.set_xscale("log")
    ax.set_xlabel(metric); ax.set_ylabel("stimulus information (bits/s)")
    return ax

plot_vs_info("isolationDistance", threshold=param["isoDmin"])
plt.show()

The dashed line is Bombcell's current threshold. Now the same plot for every metric Bombcell uses to separate good units from MUA:

In [ ]:
metrics = [  # (metric, the parameter holding its threshold, log x-axis?)
    ("isolationDistance", "isoDmin", False),
    ("Lratio", "lratioMax", True),
    ("fractionRPVs_estimatedTauR", "maxRPVviolations", False),
    ("percentageSpikesMissing_gaussian", "maxPercSpikesMissing", False),
    ("presenceRatio", "minPresenceRatio", False),
    ("nSpikes", "minNumSpikes", True),
    ("maxDriftEstimate", "maxDrift", True),
    ("spatialDecaySlope", "minSpatialDecaySlopeExp", False),
]
fig, axs = plt.subplots(2, 4, figsize=(16, 7), sharey=True)
for ax, (metric, thresh_name, log) in zip(axs.flat, metrics):
    plot_vs_info(metric, threshold=param[thresh_name], ax=ax, log=log)
plt.tight_layout(); plt.show()

✏️ **Exercise:** for each metric, where does the red curve stop rising? Is Bombcell's default threshold (dashed) in the right place
for *this* dataset, too strict (cutting into the plateau) or too lenient (letting in the rising part)?
Set your thresholds below and see how the classification changes.

In [ ]:
my_thresholds = dict(
    isoDmin=20,
    lratioMax=0.3,
    maxRPVviolations=0.1,
    maxPercSpikesMissing=20,
)
pd.DataFrame({"default": relabel(), "mine": relabel(**my_thresholds)}).fillna(0).astype(int)

How does your selection compare with the others from part 3?

In [ ]:
_, my_labels = bc.get_quality_unit_type(dict(param, **my_thresholds), quality_metrics)
mine = my_labels == "GOOD"
comparison = summary.copy()
comparison.loc["your thresholds (GOOD)"] = [mine.sum(), qm_table.loc[mine, "firing_rate"].median(),
                                            100 * qm_table.loc[mine, "responsive"].mean()]
comparison["decoding accuracy (%)"] = [100 * decode(rates[sel], stim_ids) for sel in list(selections.values()) + [mine]]
comparison.round(2)

**Discussion**
- Which metrics show a clear plateau, and which show no relationship with stimulus information at all?
- Stimulus information grows with firing rate, and so do several quality metrics. How would you disentangle the two?
- This picks thresholds for *one* question (encoding of images). For spike-timing analyses (e.g. cross-correlograms between pairs),
  contamination matters much more, so you might want stricter thresholds than the plateau suggests.

**Take-home:** pick thresholds for your question, check them against something you care about, and report them (`print_methods_text`).

## Further reading: other metrics and options

Everything above used Bombcell's defaults plus drift and distance metrics. Here are options you may want to add or change
for your own data. All are set in `param` before calling `bc.run_bombcell`.
Full list: [quality metrics and parameters](https://github.com/Julie-Fabre/bombcell/wiki/List-of-all-quality-metrics-and-parameters) ·
[how each metric works](https://github.com/Julie-Fabre/bombcell/wiki/Detailed-overview-of-quality-metrics).

**Refractory period violations: sliding refractory period** (default since Bombcell 0.94)

A unit's true refractory period is unknown and differs between cell types. Instead of assuming one value (e.g. 2 ms),
the sliding method (IBL) tests refractory periods from 1 to 10 ms and asks, with 90 % confidence, whether the
contamination is below `maxRPVviolations`. Units with too few spikes to decide either way get `RPV_status == 0`.

```python
param["rpv_method"] = "ibl_sliding"              # default; or "hill" / "llobet" with a single refractory period:
# param["rpv_method"] = "hill"; param["tauR_values"] = np.array([0.002])
param["maxRPVviolations"] = 0.1                   # max estimated contamination
param["keepUncertainRPVUnits"] = False            # True: undecidable units (RPV_status == 0) pass the RPV test
param["confidence_threshold"] = 0.9               # confidence required by the sliding method
```

**Isolation distance, L-ratio and drift**

Both are off by default because they are slower. Distance metrics use Kilosort's PC features (`pc_features.npy`)
to ask how well a unit's spikes separate from its neighbours'. Drift tracks each unit's depth over time.

```python
param["computeDistanceMetrics"] = 1               # isolationDistance, Lratio
param["isoDmin"] = 20                             # min isolation distance
param["lratioMax"] = 0.3                          # max L-ratio
param["nChannelsIsoDist"] = 4                     # nearby channels used

param["computeDrift"] = 1
param["driftBinSize"] = 60                        # s
param["maxDrift"] = 100                           # max drift (µm)
```

**Time chunks: use only the part of the recording where a unit is well isolated**

Units drift in and out, or get lost for part of a session. With time chunks, Bombcell splits the recording into windows,
computes refractory period violations and missing spikes in each, and keeps the longest stretch where the unit is good.
Metrics are then computed on that stretch only; it is stored per unit in `useTheseTimesStart` / `useTheseTimesStop`,
so you can restrict your analyses to it too.

```python
param["computeTimeChunks"] = 1
param["deltaTimeChunk"] = 360                     # window length (s)
```

✏️ **Optional:** re-run Bombcell with different options into a separate folder and compare the labels
(takes a couple of minutes here).

In [ ]:
param_alt = bc.get_default_parameters(str(ks_dir), kilosort_version=2)
param_alt.update(computeDrift=1, computeDistanceMetrics=1, computeTimeChunks=1)   # change these
qm_alt, param_alt, _, labels_alt = bc.run_bombcell(str(ks_dir), str(session_dir / "bombcell_alt"), param_alt)

pd.crosstab(pd.Series(unit_type_string, name="workshop run"), pd.Series(labels_alt, name="alternative run"))